In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [ ]:
# =========================
# 1. LOAD DATA
# =========================
df_tx = pd.read_csv("stage2_with_product_type_ml.csv", parse_dates=["order_date"])
df_cltv = pd.read_csv("stage5_rfm_cltv.csv")

In [ ]:
# =========================
# 2. PREPARE CUSTOMER FEATURES
# =========================
cust_features = df_tx.groupby("customer").agg(
    total_revenue=("revenue","sum"),
    total_qty=("qty","sum"),
    avg_order_value=("revenue","mean"),
    instrument_count=("product_type_ml", lambda x: (x=="instrument").sum()),
    sparepart_count=("product_type_ml", lambda x: (x=="sparepart").sum()),
    consumable_count=("product_type_ml", lambda x: (x=="consumable").sum()),
    last_order_date=("order_date","max")
).reset_index()

cust_features["recency_days"] = (df_tx["order_date"].max() - cust_features["last_order_date"]).dt.days

cust_features = cust_features.merge(
    df_cltv[["customer","CLTV_24m"]],
    on="customer",
    how="left"
)

cust_features = cust_features.fillna(0)

In [ ]:
# =========================
# 3. SCALING
# =========================
X = cust_features.drop(columns=["customer","last_order_date"])
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# =========================
# 4. CHOOSE K (SILHOUETTE)
# =========================
sil_scores = {}
for k in range(2,7):
    kmeans = KMeans(n_clusters=k, random_state=42)
    labels = kmeans.fit_predict(X_scaled)
    sil_scores[k] = silhouette_score(X_scaled, labels)

sil_scores

In [ ]:
# =========================
# 5. FIT FINAL MODEL
# =========================
kmeans = KMeans(n_clusters=4, random_state=42)
cust_features["cluster"] = kmeans.fit_predict(X_scaled)


In [ ]:
# =========================
# 6. PROFILE CLUSTER
# =========================
cluster_profile = cust_features.groupby("cluster").mean(numeric_only=True)
cluster_profile

In [ ]:
def map_segment(row):
    if row["total_revenue"] > cluster_profile["total_revenue"].median() and row["recency_days"] < cluster_profile["recency_days"].median():
        return "Champions"
    elif row["CLTV_24m"] > cluster_profile["CLTV_24m"].median():
        return "Growth"
    elif row["recency_days"] > cluster_profile["recency_days"].quantile(0.75):
        return "Churn"
    else:
        return "Risk"

cust_features["segment"] = cust_features.apply(map_segment, axis=1)

cust_features["segment"].value_counts()

In [ ]:
# =========================
# 7. PRODUCT FEATURES
# =========================
prod_features = df_tx.groupby(["product_name","product_type_ml"]).agg(
    total_revenue=("revenue","sum"),
    order_count=("revenue","count"),
    total_qty=("qty","sum"),
    unique_customers=("customer","nunique")
).reset_index()


In [ ]:
# =========================
# 8. SCALING
# =========================
Xp = prod_features[["total_revenue","order_count","unique_customers"]]
Xp_scaled = StandardScaler().fit_transform(Xp)

In [ ]:
# =========================
# 9. CLUSTER PRODUCT
# =========================
kmeans_p = KMeans(n_clusters=3, random_state=42)
prod_features["product_cluster"] = kmeans_p.fit_predict(Xp_scaled)

prod_profile = prod_features.groupby("product_cluster").mean(numeric_only=True)
prod_profile

In [ ]:
def map_product_segment(row):
    if row["total_revenue"] > prod_profile["total_revenue"].quantile(0.75):
        return "Big Revenue"
    elif row["order_count"] > prod_profile["order_count"].median():
        return "Potential Repeat Order"
    else:
        return "Slow Moving"

prod_features["product_segment"] = prod_features.apply(map_product_segment, axis=1)

prod_features["product_segment"].value_counts()

#TOP 20 PRODUCT BY REVENUE

In [ ]:
top20 = prod_features.sort_values("total_revenue", ascending=False).head(20)

top20[["product_name","product_type_ml","total_revenue","unique_customers","product_segment"]]


#WHO BUYS TOP PRODUCTS

In [ ]:
top_products = top20["product_name"].tolist()

buyers_top = df_tx[df_tx["product_name"].isin(top_products)]

buyer_summary = buyers_top.groupby(["product_name","customer"]).agg(
    total_revenue=("revenue","sum")
).reset_index()

buyer_summary.sample(20)


In [ ]:
cust_features.to_csv("stage6_customer_segmentation.csv", index=False)
prod_features.to_csv("stage6_product_segmentation.csv", index=False)
top20.to_csv("stage6_top20_products.csv", index=False)
